In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Subir y cargar el CSV

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving datos_supervisado_CF2P__400_m050.csv to datos_supervisado_CF2P__400_m050.csv


In [ ]:
import pandas as pd

df = pd.read_csv("datos_CF2X_800ep.csv")

print(df.head())
print(df.shape)

Preprocesar datos


In [ ]:
#Normalizamos señales y creamos ventanas de tiempo
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import json

WINDOW_SIZE = 50     # la red ve los últimos 50 pasos
BATCH_SIZE  = 128
VAL_SPLIT   = 0.2
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {DEVICE}")

INPUT_COLS = [
    'pos_x', 'pos_y', 'pos_z',
    'vel_x', 'vel_y', 'vel_z',
    'roll',  'pitch', 'yaw',
    'ang_x', 'ang_y', 'ang_z',
    'target_x', 'target_y', 'target_z',
    'err_x', 'err_y', 'err_z'
]
OUTPUT_COLS = ['motor_0', 'motor_1', 'motor_2', 'motor_3']

print(f"\nEntrada: {len(INPUT_COLS)} señales")
print(f"Salida:  {len(OUTPUT_COLS)} motores")

# ── 1. Dividir episodios ANTES de normalizar ──────────────────
episodios = df['episodio'].unique()
np.random.seed(42)
np.random.shuffle(episodios)
n_val    = int(len(episodios) * VAL_SPLIT)
ep_val   = episodios[:n_val]
ep_train = episodios[n_val:]

df_train = df[df['episodio'].isin(ep_train)].copy()
df_val   = df[df['episodio'].isin(ep_val)].copy()

# ── 2. Calcular stats SOLO con train ──────────────────────────
stats = {}
for col in INPUT_COLS + OUTPUT_COLS:
    mean = df_train[col].mean()
    std  = df_train[col].std()
    std  = std if std > 1e-8 else 1.0
    stats[col] = (float(mean), float(std))

with open('stats_normalizacion.json', 'w') as f:
    json.dump({k: list(v) for k, v in stats.items()}, f)

# ── 3. Aplicar normalización a train y val con las mismas stats
df_train_norm = df_train.copy()
df_val_norm   = df_val.copy()

for col in INPUT_COLS + OUTPUT_COLS:
    mean, std = stats[col]
    df_train_norm[col] = (df_train[col] - mean) / std
    df_val_norm[col]   = (df_val[col]   - mean) / std

print("Normalización aplicada (stats calculadas solo con train):")
print(f"  motor_0 antes: min={df_train['motor_0'].min():.0f}, max={df_train['motor_0'].max():.0f}")
print(f"  motor_0 después: min={df_train_norm['motor_0'].min():.2f}, max={df_train_norm['motor_0'].max():.2f}")
print(f"  pos_x antes: min={df_train['pos_x'].min():.2f}, max={df_train['pos_x'].max():.2f}")
print(f"  pos_x después: min={df_train_norm['pos_x'].min():.2f}, max={df_train_norm['pos_x'].max():.2f}")
print("\nStats guardadas en stats_normalizacion.json")


class DroneDataset(Dataset):
    def __init__(self, df, window_size):
        self.samples = []
        for ep in df['episodio'].unique():
            ep_df = df[df['episodio'] == ep].reset_index(drop=True)
            X = ep_df[INPUT_COLS].values.astype(np.float32)
            y = ep_df[OUTPUT_COLS].values.astype(np.float32)
            for i in range(len(X)):
                if i >= window_size:
                    window = X[i - window_size:i]
                else:
                    pad = np.zeros((window_size - i, X.shape[1]), dtype=np.float32)
                    window = np.vstack([pad, X[0:i]]) if i > 0 else np.zeros((window_size, X.shape[1]), dtype=np.float32)
                self.samples.append((window, y[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)


train_ds = DroneDataset(df_train_norm, WINDOW_SIZE)
val_ds   = DroneDataset(df_val_norm,   WINDOW_SIZE)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTrain: {len(ep_train)} episodios | {len(train_ds):,} muestras")
print(f"Val:   {len(ep_val)}  episodios  | {len(val_ds):,} muestras")
print(f"\nCada muestra tiene:")
print(f"  Entrada: {WINDOW_SIZE} pasos × {len(INPUT_COLS)} señales")
print(f"  Salida:  {len(OUTPUT_COLS)} motores")
print("\nDataset listo")

Función de entrenamiento (compartida)




In [ ]:
def entrenar_modelo(model, nombre, epochs=150, lr=1e-3):
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5
    )
    criterion = nn.MSELoss()

    train_losses = []
    val_losses   = []
    best_val     = float('inf')
    best_path    = f'modelo_{nombre.lower()}.pth'

    print(f"Entrenando {nombre}...")
    print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,} | "
          f"Épocas: {epochs} | lr: {lr} | weight_decay: 1e-5")
    print("-" * 55)

    for epoch in range(1, epochs + 1):
        model.train()
        t_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        t_loss /= len(train_dl)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                v_loss += criterion(model(xb), yb).item()
        v_loss /= len(val_dl)

        train_losses.append(t_loss)
        val_losses.append(v_loss)
        scheduler.step(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            torch.save(model.state_dict(), best_path)
            mejor = "← mejor"
        else:
            mejor = ""

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Época {epoch:3d}/{epochs} | "
                  f"Train: {t_loss:.6f} | Val: {v_loss:.6f} {mejor}")

    print("-" * 55)
    print(f"{nombre} | mejor val loss: {best_val:.6f} | guardado: {best_path}")
    return train_losses, val_losses, best_val, best_path

print("Función de entrenamiento lista")

Modelo LSTM (baseline)

In [ ]:
class LSTMDrone(nn.Module):
    def __init__(self, input_size=18, hidden_size=128,
                 num_layers=2, output_size=4, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout,
            batch_first = True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, output_size)
        )
        self.output_size = output_size

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


lstm_model = LSTMDrone(input_size=len(INPUT_COLS), output_size=len(OUTPUT_COLS))

print("Arquitectura LSTM:")
print(f"  Entrada:      {WINDOW_SIZE} pasos × {lstm_model.lstm.input_size} señales")
print(f"  hidden_size:  {lstm_model.lstm.hidden_size}")
print(f"  num_layers:   {lstm_model.lstm.num_layers}")
print(f"  Salida:       {lstm_model.output_size} motores")
print(f"  Parámetros:   {sum(p.numel() for p in lstm_model.parameters()):,}")
print()

lstm_train_losses, lstm_val_losses, lstm_best_val, lstm_path = entrenar_modelo(
    lstm_model, 'LSTM', epochs=150
)

### Modelo Mamba (mamba-ssm)

In [ ]:
!pip install "mamba-ssm[causal-conv1d]" --no-build-isolation -q

In [ ]:
from mamba_ssm import Mamba

class MambaDrone(nn.Module):
    def __init__(self, input_size=18, d_model=128,
                 n_layers=2, d_state=16, d_conv=4,
                 expand=2, output_size=4):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.mamba_layers = nn.ModuleList([
            Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_layers)
        ])
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, output_size)
        )

    def forward(self, x):
        x = self.input_proj(x)
        for mamba, norm in zip(self.mamba_layers, self.norms):
            x = norm(x + mamba(x))
        return self.fc(x[:, -1, :])


mamba_model = MambaDrone(input_size=len(INPUT_COLS), output_size=len(OUTPUT_COLS))

print("Arquitectura Mamba:")
print(f"  Entrada:    {WINDOW_SIZE} pasos × {len(INPUT_COLS)} señales")
print(f"  d_model:    128")
print(f"  n_layers:   2")
print(f"  d_state:    16  |  d_conv: 4  |  expand: 2")
print(f"  Salida:     {len(OUTPUT_COLS)} motores")
print(f"  Parámetros: {sum(p.numel() for p in mamba_model.parameters()):,}")
print()

mamba_train_losses, mamba_val_losses, mamba_best_val, mamba_path = entrenar_modelo(
    mamba_model, 'Mamba', epochs=150
)

## Descargar modelos entrenados

In [ ]:
from google.colab import files

files.download('modelo_lstm.pth')
files.download('modelo_mamba.pth')
files.download('stats_normalizacion.json')